# Генерация данных: тестирование детей и банк заданий

Этот ноутбук генерирует все необходимые данные и сохраняет их в Excel-файлы:
- **children_data.xlsx** — результаты тестирования 300 детей
- **question_meta.xlsx** — описание 50 диагностических заданий
- **task_bank.xlsx** — банк развивающих заданий (4 уровня × 5 категорий)


In [9]:
import numpy as np
import pandas as pd

np.random.seed(42)


## Параметры генерации


In [10]:
N_CHILDREN  = 300
N_QUESTIONS = 50

# Демографика
ages    = np.random.choice([3, 4, 5, 6, 7], size=N_CHILDREN,
                            p=[0.12, 0.23, 0.30, 0.22, 0.13])
genders = np.random.choice(['М', 'Ж'], size=N_CHILDREN)
ids     = [f'CHILD_{i:04d}' for i in range(1, N_CHILDREN + 1)]

# Латентный уровень развития (используется только для реалистичной генерации)
age_factor        = (ages - 3) / 4
individual_factor = np.random.normal(0, 0.18, N_CHILDREN)
true_score        = np.clip(age_factor + individual_factor, 0, 1)

print(f'Параметры: {N_CHILDREN} детей, {N_QUESTIONS} заданий')


Параметры: 300 детей, 50 заданий


## Функции генерации ответов


In [11]:
def binary_responses(scores, base_prob, n):
    '''Бинарные ответы (0/1), вероятность растёт с уровнем.'''
    R = np.zeros((len(scores), n))
    for i, s in enumerate(scores):
        p = np.clip(base_prob + s * (1 - base_prob) * 0.85, 0.08, 0.96)
        R[i] = np.random.binomial(1, p, n)
    return R

def count_responses(scores, n, max_score=3):
    '''Балльные ответы (0..max_score).'''
    R = np.zeros((len(scores), n))
    for i, s in enumerate(scores):
        mu = s * max_score
        for j in range(n):
            val = int(np.round(np.clip(np.random.normal(mu, 0.65), 0, max_score)))
            R[i, j] = val
    return R


## Генерация ответов по категориям


In [12]:
shapes   = binary_responses(true_score, 0.30, 10)
colors   = binary_responses(true_score, 0.40, 10)
counting = count_responses(true_score, 10, max_score=3)
memory   = binary_responses(true_score, 0.22, 10)
logic    = binary_responses(true_score, 0.18, 10)

q_cols = (
    [f'Форма_{i}'  for i in range(1, 11)] +
    [f'Цвет_{i}'   for i in range(1, 11)] +
    [f'Счёт_{i}'   for i in range(1, 11)] +
    [f'Память_{i}' for i in range(1, 11)] +
    [f'Логика_{i}' for i in range(1, 11)]
)

df = pd.DataFrame(
    np.hstack([shapes, colors, counting, memory, logic]).astype(int),
    columns=q_cols
)
df.insert(0, 'ID',      ids)
df.insert(1, 'Возраст', ages)
df.insert(2, 'Пол',     genders)

print(f'Датасет сформирован: {df.shape[0]} строк × {df.shape[1]} столбцов')
df.head()


Датасет сформирован: 300 строк × 53 столбцов


,ID,Возраст,Пол,Форма_1,Форма_2,Форма_3,Форма_4,Форма_5,Форма_6,Форма_7,...,Логика_1,Логика_2,Логика_3,Логика_4,Логика_5,Логика_6,Логика_7,Логика_8,Логика_9,Логика_10
0,CHILD_0001,5,М,1,0,1,0,1,0,1,...,1,0,0,0,0,1,0,0,1,1
1,CHILD_0002,7,Ж,1,1,1,1,1,0,1,...,1,1,1,1,1,1,1,1,1,1
2,CHILD_0003,6,Ж,1,1,1,1,1,1,1,...,1,1,1,0,1,1,1,1,1,1
3,CHILD_0004,5,Ж,0,0,1,1,0,1,1,...,0,0,1,0,1,1,0,0,1,1
4,CHILD_0005,4,М,0,0,0,1,0,0,1,...,0,0,1,0,0,1,0,0,1,1


## Сохранение в Excel


In [13]:
df.to_excel('children_data.xlsx', index=False)
print('Данные сохранены: children_data.xlsx')
print(f'  Строк: {len(df)}')
print(f'  Столбцов: {len(df.columns)}')
print(f'  Столбцы: ID, Возраст, Пол + {N_QUESTIONS} заданий')


Данные сохранены: children_data.xlsx
  Строк: 300
  Столбцов: 53
  Столбцы: ID, Возраст, Пол + 50 заданий


## 1. Метаданные диагностических заданий


In [14]:
QUESTION_META = {
    # Формы (1-10)
    'Форма_1':  'Покажи на картинке круг',
    'Форма_2':  'Покажи на картинке треугольник',
    'Форма_3':  'Покажи на картинке квадрат',
    'Форма_4':  'Обведи все прямоугольники на рисунке',
    'Форма_5':  'Назови форму предмета (мяч)',
    'Форма_6':  'Назови форму предмета (книга)',
    'Форма_7':  'Сколько углов у треугольника?',
    'Форма_8':  'Найди одинаковые фигуры среди 6 предложенных',
    'Форма_9':  'Собери фигуру из частей (пазл)',
    'Форма_10': 'Нарисуй дом, используя квадрат и треугольник',
    # Цвета (11-20)
    'Цвет_1':  'Покажи красный предмет',
    'Цвет_2':  'Покажи синий предмет',
    'Цвет_3':  'Назови цвет солнца',
    'Цвет_4':  'Назови цвет травы',
    'Цвет_5':  'Какого цвета небо в ясный день?',
    'Цвет_6':  'Найди два предмета одного цвета среди 5',
    'Цвет_7':  'Разложи карточки по цвету (3 группы)',
    'Цвет_8':  'Что получится, если смешать красный и синий?',
    'Цвет_9':  'Расставь оттенки от светлого к тёмному',
    'Цвет_10': 'Нарисуй радугу и назови все цвета',
    # Счёт (21-30, балл 0-3)
    'Счёт_1':  'Посчитай звёздочки (1-3): 0=неверно, 1=±1, 2=±0, 3=точно',
    'Счёт_2':  'Посчитай яблоки (1-5)',
    'Счёт_3':  'Посчитай машинки (1-7)',
    'Счёт_4':  'Покажи карточку с числом 3',
    'Счёт_5':  'Покажи карточку с числом 5',
    'Счёт_6':  'Что больше: 2 или 4?',
    'Счёт_7':  'Положи 4 кубика',
    'Счёт_8':  'Сколько пальцев на одной руке?',
    'Счёт_9':  'Сколько ног у кошки?',
    'Счёт_10': 'Реши задачку: было 3 яблока, добавили 2. Сколько стало?',
    # Память (31-40)
    'Память_1':  'Запомни 2 предмета, назови их через 30 сек.',
    'Память_2':  'Запомни 3 предмета, назови их через 30 сек.',
    'Память_3':  'Повтори последовательность из 3 хлопков',
    'Память_4':  'Повтори последовательность из 4 хлопков',
    'Память_5':  'Запомни короткий рассказ, перескажи',
    'Память_6':  'Найди изменение: какого предмета не стало (из 4)?',
    'Память_7':  'Найди изменение: какого предмета не стало (из 6)?',
    'Память_8':  'Повтори ряд из 3 цифр',
    'Память_9':  'Повтори ряд из 4 цифр',
    'Память_10': 'Запомни 5 картинок, укажи их среди 10',
    # Логика (41-50)
    'Логика_1':  'Что лишнее среди 4 предметов (по форме)?',
    'Логика_2':  'Что лишнее среди 4 предметов (по цвету)?',
    'Логика_3':  'Продолжи ряд фигур (АББАБ…)',
    'Логика_4':  'Продолжи ряд фигур (сложнее)',
    'Логика_5':  'Сортировка по размеру (3 предмета)',
    'Логика_6':  'Сортировка по размеру (5 предметов)',
    'Логика_7':  'Найди пару (ложка-вилка, шапка-варежки…)',
    'Логика_8':  'Причина и следствие: почему намок зонтик?',
    'Логика_9':  'Разгадай загадку',
    'Логика_10': 'Реши задачу на простую аналогию',
}

import pandas as pd
meta_df = pd.DataFrame.from_dict(QUESTION_META, orient='index', columns=['Описание задания'])
meta_df.index.name = 'Задание'
meta_df.to_excel('question_meta.xlsx')
print(f'Сохранено {len(QUESTION_META)} заданий → question_meta.xlsx')
meta_df


Сохранено 50 заданий → question_meta.xlsx


,Описание задания
Задание,
Форма_1,Покажи на картинке круг
Форма_2,Покажи на картинке треугольник
Форма_3,Покажи на картинке квадрат
Форма_4,Обведи все прямоугольники на рисунке
Форма_5,Назови форму предмета (мяч)
Форма_6,Назови форму предмета (книга)
Форма_7,Сколько углов у треугольника?
Форма_8,Найди одинаковые фигуры среди 6 предложенных
Форма_9,Собери фигуру из частей (пазл)


## 2. Банк развивающих заданий


In [15]:
TASK_BANK = {
    0: {  # Уровень А: Требует поддержки
        'forms': [
            'Обведи пальцем круг на карточке, затем нарисуй его в воздухе',
            'Найди все круглые предметы в комнате и разложи их на столе',
            'Сложи аппликацию «домик» из готовых фигур: квадрат + треугольник',
            'Раскрась только треугольники на листе (остальные фигуры не трогай)',
            'Скажи, на какую фигуру похож арбуз? Колесо? Книжка?',
            'Обведи по контуру квадрат и раскрась его синим',
            'Выбери из кучки все квадратики (дана смесь кругов/квадратов/треугольников)',
            'Посмотри на фигуру-образец, найди такую же среди 4 предложенных',
        ],
        'colors': [
            'Покажи на карточке красный цвет — потрогай, назови',
            'Разложи кубики: красные в одну коробку, синие — в другую',
            'Назови цвет каждого предмета на картинке (3 предмета)',
            'Найди в комнате 2 предмета жёлтого цвета',
            'Раскрась солнышко жёлтым карандашом',
            'Покажи зелёный предмет среди 4 разноцветных',
            'Подбери пару по цвету (носочки: 2 красных, 2 синих)',
            'Нарисуй траву зелёным и небо синим',
        ],
        'counting': [
            'Посчитай вслух яблоки на картинке (1–3 яблока)',
            'Положи столько же кубиков, сколько показывает карточка с точками (1–2)',
            'Сколько пальцев ты показываешь? Покажи 2, потом 3',
            'Хлопни в ладоши столько раз, сколько звёздочек (1–3)',
            'Посчитай ступеньки на рисунке лестницы (2–4 ступеньки)',
            'Разложи 3 карандаша по одному и назови: «один, два, три»',
            'Покажи на пальцах число 2',
            'Выбери карточку с цифрой 1, затем с цифрой 2',
        ],
        'memory': [
            'Запомни 2 предмета на столе, закрой глаза — какой убрали?',
            'Повтори за педагогом: «хлоп-хлоп» (2 хлопка)',
            'Запомни и покажи: сначала мяч, потом кубик',
            'Посмотри на 2 картинки, переверни — назови что видел',
            'Повтори простую фразу: «У кошки мягкие лапки»',
            'Спрячь 2 игрушки под платок — какие спрятаны?',
            'Запомни цвет карточки, найди такой же цвет в комнате',
            'Повтори движение: поднять руки, затем хлопнуть',
        ],
        'logic': [
            'Что лишнее: яблоко, груша, машинка, банан? (по смыслу)',
            'Продолжи ряд: круг–квадрат–круг–…? (нарисуй)',
            'Разложи игрушки: большие отдельно, маленькие отдельно',
            'Найди пару: ложка и … (вилка / мяч / книга)?',
            'Что бывает зимой: снег или цветы?',
            'Что нужно для рисования: карандаш или ботинок?',
            'Посмотри на ряд: солнышко–тучка–солнышко–… Что дальше?',
            'Помоги Мишке найти свой домик (лабиринт из 2 поворотов)',
        ],
    },
    1: {  # Уровень Б: Средний уровень
        'forms': [
            'Нарисуй портрет человека, используя круги и овалы',
            'Сколько треугольников в рисунке? Посчитай и запиши',
            'Сложи из спичек квадрат, затем треугольник',
            'Найди все прямоугольники в классной комнате, запиши сколько',
            'Нарисуй машину, используя только прямоугольники и круги',
            'Раздели квадрат на 2 равных треугольника (нарисуй линию)',
            'Определи: какая фигура симметрична? (из 3 вариантов)',
            'Собери танграм из 4 фигур по образцу',
        ],
        'colors': [
            'Назови все цвета радуги по порядку',
            'Что получится, если смешать жёлтый и красный? Проверь красками',
            'Расставь карточки от самого светлого к самому тёмному (5 оттенков)',
            'Нарисуй закат: используй красный, оранжевый, жёлтый',
            'Найди в журнале 3 предмета одного цвета, вырежи и наклей',
            'Назови 5 предметов зелёного цвета в природе',
            'Раскрась картинку по инструкции (небо=синий, трава=зелёный…)',
            'Придумай и нарисуй животное, которого не существует — выбери 3 любых цвета',
        ],
        'counting': [
            'Посчитай предметы в комнате по группам (стулья, окна, двери)',
            'Реши задачу: было 4 яблока, съели 2 — сколько осталось?',
            'Запиши цифры от 1 до 7 в ряд',
            'Сколько ног у 2 кошек?',
            'Положи 5 кубиков, убери 2 — сколько осталось?',
            'Что больше: 3 или 5? Покажи на пальцах',
            'Посчитай слоги в словах: «ко-шка», «со-ба-ка»',
            'Реши: 2 + 3 = ? (используй кубики)',
        ],
        'memory': [
            'Запомни 4 предмета, назови через 1 минуту',
            'Повтори последовательность из 5 хлопков',
            'Запомни короткий стих (2 строчки) и повтори',
            'Запомни 3 слова: «кот, мяч, дом» — назови в обратном порядке',
            'Посмотри 30 сек. на картинку, расскажи что видел',
            'Запомни и нарисуй: треугольник–квадрат–круг',
            'Повтори ряд: «красный–синий–жёлтый» (по карточкам)',
            'Найди изменение: убрали 1 предмет из 5 — какой?',
        ],
        'logic': [
            'Что лишнее: роза, ромашка, берёза, тюльпан?',
            'Продолжи ряд: 1–3–5–…–… (закономерность +2)',
            'Как связаны: птица–гнездо, рыба–…?',
            'Расставь картинки «Утро–День–Вечер–Ночь» по порядку',
            'Решение задачи: почему листья упали? (осень/ветер/…)',
            'Найди ошибку на картинке (рыба летит, рыбка в дереве…)',
            'Продолжи паттерн: АБВ–АБВ–А…',
            'Что сначала: яйцо или курица? Обоснуй',
        ],
    },
    2: {  # Уровень В: Уверенный уровень
        'forms': [
            'Нарисуй план своей комнаты (вид сверху) с обозначением мебели',
            'Объясни разницу между ромбом и квадратом (свойства углов и сторон)',
            'Создай орнамент из 3 видов фигур, соблюдая ритм',
            'Сколько сторон и углов у пятиугольника? Нарисуй его',
            'Разрежь прямоугольник на 4 равных части двумя способами',
            'Придумай и нарисуй существо из 7 геометрических фигур',
            'Определи периметр прямоугольника 4×3 клетки (посчитай клетки)',
            'Составь фигуру из 5 спичек по схеме',
        ],
        'colors': [
            'Объясни, что такое тёплые и холодные цвета, приведи примеры',
            'Нарисуй натюрморт из 3 фруктов: передай объём цветом',
            'Смешай 2 краски и угадай что получится (6 пар)',
            'Найди 3 картины художников, где преобладает синий — объясни настроение',
            'Создай цветовое колесо из 6 основных и составных цветов',
            'Раскрась картинку в определённой цветовой гамме (только холодные тона)',
            'Объясни: почему листья зелёные летом и жёлтые осенью?',
            'Нарисуй пейзаж «утро» и «вечер» — покажи разницу в цвете',
        ],
        'counting': [
            'Реши примеры: 7+4=?, 12–5=?, 3×3=?',
            'Измерь длину стола карандашами (условная мерка)',
            'Запиши числа от 10 до 1 в убывающем порядке',
            'Реши задачу: в 3 коробках по 4 карандаша — сколько всего?',
            'Сравни дроби: что больше — половина яблока или четверть?',
            'Посчитай: сколько чётных чисел от 1 до 10?',
            'Реши: 15–7=? используй счёты или пальцы',
            'Составь задачу на сложение сам и реши её',
        ],
        'memory': [
            'Запомни 6 предметов, назови через 2 минуты',
            'Перескажи сказку «Репка» по памяти в правильном порядке',
            'Запомни 5 цифр и напиши их через 1 минуту',
            'Запомни и воспроизведи ритмический рисунок (7 элементов)',
            'Запомни 5 слов, придумай историю с ними',
            'Посмотри 20 сек. на карту района, нарисуй по памяти',
            'Запомни 4 правила игры и выполни их по порядку',
            'Повтори стихотворение (4 строчки) с первого раза',
        ],
        'logic': [
            'Реши ребус: слово с картинками (3 слога)',
            'Продолжи последовательность: 2–4–8–16–…',
            'Разгадай загадку-описание (3 признака)',
            'Составь рассказ по серии из 5 картинок',
            'Решение логической задачи: «У Пети есть кот. У кота 4 лапы…»',
            'Найди 5 отличий между двумя картинками',
            'Классифицируй 12 карточек на 3 группы самостоятельно',
            'Объясни: почему круглое колесо лучше квадратного?',
        ],
    },
    3: {  # Уровень Г: Продвинутый уровень
        'forms': [
            'Исследуй: сколько граней, вершин, рёбер у куба? Запиши формулой',
            'Создай 3D-модель дома из бумаги (оригами)',
            'Объясни понятие симметрии и найди 5 симметричных объектов в природе',
            'Нарисуй перспективу: дорога, уходящая вдаль',
            'Составь геометрическое доказательство: почему сумма углов треугольника 180°?',
            'Создай орнамент с поворотной симметрией 4-го порядка',
            'Реши задачу: разрежь Г-фигуру на 2 прямоугольника — найди площадь каждого',
            'Придумай и запиши алгоритм рисования правильного пятиугольника',
        ],
        'colors': [
            'Изучи закон Гёте о цвете — расскажи об одном открытии',
            'Создай абстрактную картину, передающую эмоцию «радость» только цветом',
            'Смешай краски и создай 5 собственных оттенков, придумай им названия',
            'Объясни, как художники используют цветовой контраст для выразительности',
            'Нарисуй мандалу, используя симметрию и не менее 6 цветов',
            'Исследуй: какие цвета использовал Ван Гог и почему?',
            'Создай «цветовой дневник»: 5 дней — 5 цветов настроения с рисунками',
            'Спроектируй логотип для детского клуба: обоснуй выбор цветов',
        ],
        'counting': [
            'Реши: 48 ÷ 6 =?, 7 × 8 =?, 100 – 37 =?',
            'Задача: «Поезд идёт 2 часа со скоростью 60 км/ч — сколько проехал?»',
            'Найди все простые числа от 1 до 30',
            'Запиши число 35 в разных системах: тройками, пятёрками, десятками',
            'Реши уравнение: X + 12 = 20',
            'Задача на дроби: «Съел 1/3 торта, осталось 2/3 — это сколько кусочков из 9?»',
            'Придумай и реши задачу с тремя действиями',
            'Исследуй числа Фибоначчи: продолжи ряд до 10-го элемента',
        ],
        'memory': [
            'Запомни 10 слов за 30 сек., воспроизведи через 5 минут',
            'Метод Цицерона: запомни 7 предметов, «разложи» по комнатам дома',
            'Запомни стихотворение (8 строк) за 15 минут',
            'Запомни и воспроизведи последовательность 9 цифр',
            'Игра «Снежный ком»: добавляй слово к цепочке (до 10 слов)',
            'Запомни правила настольной игры (7+ пунктов) и объясни другу',
            'Техника «Ассоциации»: запомни 8 пар слов (кот–звезда, дом–яблоко…)',
            'Нарисуй по памяти карту своего района с 5+ деталями',
        ],
        'logic': [
            'Реши логическую задачу Эйнштейна (упрощённую, 3 условия)',
            'Создай собственную загадку с тремя признаками',
            'Придумай правила новой настольной игры для 3 игроков',
            'Анализ ситуации: «Все кошки — животные. Мурка — кошка. Значит…»',
            'Составь алгоритм (блок-схему) «Как приготовить бутерброд»',
            'Найди паттерн: 1, 4, 9, 16, … — что это? Продолжи до 7 элементов',
            'Придумай 3 способа использования обычного кирпича (нестандартное мышление)',
            'Создай и реши свою математическую задачу для детей уровня Б',
        ],
    },
}


In [16]:
import pandas as pd

rows = []
for lvl in range(4):
    for cat, tasks in TASK_BANK[lvl].items():
        for task in tasks:
            rows.append({'Уровень': lvl, 'Категория': cat, 'Задание': task})

task_bank_df = pd.DataFrame(rows)
task_bank_df.to_excel('task_bank.xlsx', index=False)

print('Банк заданий сохранён: task_bank.xlsx')
for lvl in range(4):
    n = len(task_bank_df[task_bank_df['Уровень'] == lvl])
    print(f'  Уровень {lvl}: {n} заданий')
print(f'  Итого: {len(task_bank_df)} заданий')
task_bank_df.head(10)


Банк заданий сохранён: task_bank.xlsx
  Уровень 0: 40 заданий
  Уровень 1: 40 заданий
  Уровень 2: 40 заданий
  Уровень 3: 40 заданий
  Итого: 160 заданий


,Уровень,Категория,Задание
0,0,forms,"Обведи пальцем круг на карточке, затем нарисуй..."
1,0,forms,Найди все круглые предметы в комнате и разложи...
2,0,forms,Сложи аппликацию «домик» из готовых фигур: ква...
3,0,forms,Раскрась только треугольники на листе (остальн...
4,0,forms,"Скажи, на какую фигуру похож арбуз? Колесо? Кн..."
5,0,forms,Обведи по контуру квадрат и раскрась его синим
6,0,forms,Выбери из кучки все квадратики (дана смесь кру...
7,0,forms,"Посмотри на фигуру-образец, найди такую же сре..."
8,0,colors,"Покажи на карточке красный цвет — потрогай, на..."
9,0,colors,"Разложи кубики: красные в одну коробку, синие ..."
